In [1]:
!pip install --upgrade pip --quiet
!pip install --upgrade diffusers transformers accelerate controlnet-aux datasets --quiet
!pip install --upgrade peft --quiet
!pip install torch-fidelity lpips --quiet

In [2]:
import torch
from diffusers import StableDiffusionAdapterPipeline, T2IAdapter, UniPCMultistepScheduler
from diffusers.utils import load_image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import os
from PIL import Image
import numpy as np
import lpips
from tqdm import tqdm
from torch.cuda.amp import GradScaler, autocast 
from torch_fidelity import calculate_metrics
import warnings
warnings.filterwarnings("ignore")

2025-11-17 10:29:21.410797: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763375361.433382     260 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763375361.440136     260 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [3]:
base_dir = "/kaggle/input/face-sketches-collection/small_hed-augmented_ffhq_dataset"
test_dir = base_dir 
num_epochs = 20
batch_size = 4 
accumulation_steps = 8
effective_batch_size = batch_size * accumulation_steps
device = "cuda"
# prompt = "a realistic photo of a human face"
prompt = """(hyper-realistic photo:1.2), (ultra-detailed skin texture:1.1), 
            detailed pores, realistic eyes, sharp focus, 
            8k UHD, professional studio lighting, DSLR"""

adapter_name = "TencentARC/t2iadapter_sketch_sd15v2"
stable_diff_name = "botp/stable-diffusion-v1-5"
padding = "max_length"
return_tensors="pt"
scaler = GradScaler()
patience = 5
best_eval_loss = float('inf')
best_model_path = "/kaggle/working/adapter_best_model"
latest_model_path = "/kaggle/working/adapter_latest_model"
generated_dir = "/kaggle/working/generated_for_metrics"
max_eval_samples = 500 

# Dataset


In [4]:
class SketchToPhotoDataset(Dataset):
    def __init__(self, hed_dir, photo_dir, max_samples=None):
        self.hed_dir = hed_dir
        self.photo_dir = photo_dir
        self.hed_files = sorted(os.listdir(hed_dir))
        self.photo_files = sorted(os.listdir(photo_dir))
        
        if max_samples and len(self.hed_files) > max_samples:
            import random
            indices = random.sample(range(len(self.hed_files)), max_samples)
            self.hed_files = [self.hed_files[i] for i in indices]
            self.photo_files = [self.photo_files[i] for i in indices]
        
        self.condition_transform = transforms.Compose([
            transforms.Resize((512, 512)),
            transforms.ToTensor()  
        ])
        self.target_transform = transforms.Compose([
            transforms.Resize((512, 512)),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]) 
        ])
    
    def __len__(self):
        return len(self.hed_files)

    def __getitem__(self, idx):
        hed_path = os.path.join(self.hed_dir, self.hed_files[idx])
        photo_path = os.path.join(self.photo_dir, self.photo_files[idx])
        
        hed_image = Image.open(hed_path).convert("L")
        photo = Image.open(photo_path).convert("RGB")
        
        hed_image = self.condition_transform(hed_image)  
        photo = self.target_transform(photo) 
        
        return {"hed": hed_image, "photo": photo}

In [5]:
train_dataset = SketchToPhotoDataset(
    hed_dir=os.path.join(base_dir, "train", "sketches"),
    photo_dir=os.path.join(base_dir, "train", "photos"),
    max_samples=None 
)
val_dataset = SketchToPhotoDataset(
    hed_dir=os.path.join(base_dir, "val", "sketches"),
    photo_dir=os.path.join(base_dir, "val", "photos"),
    max_samples=None 
)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Models

In [6]:
adapter = T2IAdapter.from_pretrained(
    adapter_name,  
)

pipe = StableDiffusionAdapterPipeline.from_pretrained(
    stable_diff_name, 
    adapter=adapter, 
    dtype=torch.float16
)

pipe.unet.requires_grad_(False)
pipe.text_encoder.requires_grad_(False)
pipe.vae.requires_grad_(False)

adapter.to(device) 
adapter.requires_grad_(True) 

pipe.to(device) 

optimizer = torch.optim.AdamW(adapter.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=1000)

An error occurred while trying to fetch TencentARC/t2iadapter_sketch_sd15v2: TencentARC/t2iadapter_sketch_sd15v2 does not appear to have a file named diffusion_pytorch_model.safetensors.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
Keyword arguments {'dtype': torch.float16} are not expected by StableDiffusionAdapterPipeline and will be ignored.


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

# Training

In [7]:
patience_counter = 0

for epoch in range(num_epochs):
    adapter.train() 
    epoch_loss = 0
    
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch} Training")
    
    for step, batch in enumerate(progress_bar):
        hed_images = batch["hed"].to(device)
        photos = batch["photo"].to(device)
        
        with autocast():
            with torch.no_grad():
                latents = pipe.vae.encode(photos).latent_dist.sample() * pipe.vae.config.scaling_factor
            
            bsz = latents.shape[0]
            timesteps = torch.randint(0, pipe.scheduler.config.num_train_timesteps, (bsz,), device=device).long()
            noise = torch.randn_like(latents)
            noisy_latents = pipe.scheduler.add_noise(latents, noise, timesteps)
            
            use_null_prompt = torch.rand(1).item() < 0.1
            
            if use_null_prompt:
                final_prompt = "" 
            else:
                final_prompt = prompt
                
            text_inputs = pipe.tokenizer(
                final_prompt, 
                padding=padding, 
                max_length=pipe.tokenizer.model_max_length, 
                truncation=True, 
                return_tensors=return_tensors
            )
        
            text_input_ids = text_inputs.input_ids.to(device)
            
            with torch.no_grad():
                encoder_hidden_states = pipe.text_encoder(text_input_ids)[0]
                if encoder_hidden_states.shape[0] != bsz:
                    encoder_hidden_states = encoder_hidden_states.repeat(bsz, 1, 1)

            # Forward Adapter
            adapter_features = adapter(hed_images) 
            
            # Forward UNet
            noise_pred = pipe.unet(
                noisy_latents, 
                timestep=timesteps, 
                encoder_hidden_states=encoder_hidden_states, 
                down_intrablock_additional_residuals=list(adapter_features), 
                
            ).sample
            

            loss = torch.nn.functional.mse_loss(noise_pred, noise)
            epoch_loss += loss.item()
            
            loss = loss / accumulation_steps
        
        scaler.scale(loss).backward()
        
        if (step + 1) % accumulation_steps == 0:
            scaler.step(optimizer)
            scaler.update() 
            optimizer.zero_grad()
        
        progress_bar.set_postfix(Loss=f"{loss.item() * accumulation_steps:.4f}")
    
    avg_train_loss = epoch_loss / len(train_dataloader)
    print(f"\nEpoch {epoch}, Avg Train Loss: {avg_train_loss:.4f}")

    # Validation
    adapter.eval()
    val_loss = 0
    val_progress_bar = tqdm(val_dataloader, desc=f"Epoch {epoch} Validation")
    
    with torch.no_grad():
        for batch in val_progress_bar:
            hed_images = batch["hed"].to(device)
            photos = batch["photo"].to(device)
            
            latents = pipe.vae.encode(photos).latent_dist.sample() * pipe.vae.config.scaling_factor
            bsz = latents.shape[0]
            timesteps = torch.randint(0, pipe.scheduler.config.num_train_timesteps, (bsz,), device=device).long()
            noise = torch.randn_like(latents)
            noisy_latents = pipe.scheduler.add_noise(latents, noise, timesteps)
            
            text_inputs = pipe.tokenizer(
                prompt, 
                padding=padding, 
                max_length=pipe.tokenizer.model_max_length, 
                truncation=True, 
                return_tensors=return_tensors
            )
            
            text_input_ids = text_inputs.input_ids.to(device)
            
            encoder_hidden_states = pipe.text_encoder(text_input_ids)[0]
            
            if encoder_hidden_states.shape[0] != bsz:
                encoder_hidden_states = encoder_hidden_states.repeat(bsz, 1, 1)

            # Forward Adapter
            adapter_features = adapter(hed_images)
            
            # Forward UNet
            noise_pred = pipe.unet(
                noisy_latents, 
                timestep=timesteps, 
                encoder_hidden_states=encoder_hidden_states, 
                down_intrablock_additional_residuals=list(adapter_features), 
                
            ).sample
            
            val_loss += torch.nn.functional.mse_loss(noise_pred, noise).item()
            
            val_progress_bar.set_postfix(Val_Loss=f"{val_loss / len(val_dataloader):.4f}")
            
    avg_val_loss = val_loss / len(val_dataloader)
    print(f"Epoch {epoch}, Avg Val Loss: {avg_val_loss:.4f}")
    
    # Early stopping
    if avg_val_loss < best_eval_loss:
        best_eval_loss = avg_val_loss
        patience_counter = 0
        
        adapter.save_pretrained(best_model_path)
        print(f"Saved best model at: {best_model_path}")
        
    else:
        patience_counter += 1
        print(f"Patience: {patience_counter} / {patience}")
        
        if patience_counter >= patience:
            print(f"Early stopping after {patience} epochs.")
            break 
    
    scheduler.step()

adapter.save_pretrained(latest_model_path)
print(f"Saved best model (Eval Loss: {best_eval_loss:.4f}) at: {best_model_path}")
print(f"Saved final model at: {latest_model_path}")

Epoch 0 Training: 100%|██████████| 707/707 [25:33<00:00,  2.17s/it, Loss=0.0886]



Epoch 0, Avg Train Loss: 0.1359


Epoch 0 Validation: 100%|██████████| 40/40 [00:43<00:00,  1.08s/it, Val_Loss=0.1330]


Epoch 0, Avg Val Loss: 0.1330
Saved best model at: /kaggle/working/adapter_best_model


Epoch 1 Training: 100%|██████████| 707/707 [24:34<00:00,  2.09s/it, Loss=0.0496]


...

Epoch 9, Avg Val Loss: 0.1339
Patience: 4 / 5


Epoch 10 Training: 100%|██████████| 707/707 [24:34<00:00,  2.09s/it, Loss=0.2116]



Epoch 10, Avg Train Loss: 0.1325


Epoch 10 Validation: 100%|██████████| 40/40 [00:39<00:00,  1.02it/s, Val_Loss=0.1407]


Epoch 10, Avg Val Loss: 0.1407
Patience: 5 / 5
Early stopping after 5 epochs.
Saved best model (Eval Loss: 0.1189) at: /kaggle/working/adapter_best_model
Saved final model at: /kaggle/working/adapter_latest_model


In [8]:
!zip -r -q /kaggle/working/adapter_best_model.zip /kaggle/working/adapter_best_model

# Testing 

In [9]:
real_dir = os.path.join(test_dir, "test", "photos")
sketch_dir = os.path.join(test_dir, "test", "sketches")

os.makedirs(generated_dir, exist_ok=True)

generator = torch.Generator(device=device).manual_seed(1234)

In [10]:
prompt = """(hyper-realistic photo:1.2), (ultra-detailed skin texture:1.1), 
            detailed pores, realistic eyes, sharp focus, 
            8k UHD, professional studio lighting, DSLR"""

negative_prompt = """(drawing:1.4), (sketch:1.4), (painting:1.3), cartoon, 3D, 
                    render, CGI, anime, illustration, (deformed:1.2), (disfigured:1.2), 
                    ugly, bad anatomy, (blurry:1.1), low quality, low-res"""

In [11]:
adapter = T2IAdapter.from_pretrained(
    best_model_path, 
    torch_dtype=torch.float16
)

pipe = StableDiffusionAdapterPipeline.from_pretrained(
    stable_diff_name, 
    adapter=adapter, 
    dtype=torch.float16
)

pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe.to(device)

loss_fn_vgg = lpips.LPIPS(net='vgg').to(device)

Keyword arguments {'dtype': torch.float16} are not expected by StableDiffusionAdapterPipeline and will be ignored.


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth
100%|██████████| 528M/528M [00:02<00:00, 210MB/s] 


Loading model from: /usr/local/lib/python3.11/dist-packages/lpips/weights/v0.1/vgg.pth


### LIPIPS

In [12]:
lpips_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

total_lpips_distance = 0
image_count = 0

sketch_files = sorted(os.listdir(sketch_dir))

In [13]:
for i, filename in enumerate(tqdm(sketch_files)):
    if max_eval_samples and i >= max_eval_samples:
        break

    sketch_path = os.path.join(sketch_dir, filename)
    real_path = os.path.join(real_dir, filename)
    generated_path = os.path.join(generated_dir, filename)

    condition_image = load_image(sketch_path).convert("L").resize((512, 512))

    generated_image_pil = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=condition_image,
        num_inference_steps=30,
        generator=generator,
        guidance_scale=7.5,
        adapter_conditioning_scale=0.9 
    ).images[0]
    
    generated_image_pil.save(generated_path)
    real_image_pil = load_image(real_path)
    real_tensor = lpips_transform(real_image_pil).to(device)
    gen_tensor = lpips_transform(generated_image_pil).to(device)

    with torch.no_grad():
        dist = loss_fn_vgg(real_tensor.unsqueeze(0), gen_tensor.unsqueeze(0))
    
    total_lpips_distance += dist.item()
    image_count += 1

  0%|          | 0/158 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  1%|          | 1/158 [00:08<21:53,  8.37s/it]

  0%|          | 0/30 [00:00<?, ?it/s]

  1%|▏         | 2/158 [00:16<21:25,  8.24s/it]

...

 99%|█████████▊| 156/158 [21:12<00:16,  8.15s/it]

  0%|          | 0/30 [00:00<?, ?it/s]

 99%|█████████▉| 157/158 [21:20<00:08,  8.15s/it]

  0%|          | 0/30 [00:00<?, ?it/s]

100%|██████████| 158/158 [21:28<00:00,  8.15s/it]


In [14]:
avg_lpips = total_lpips_distance / image_count

print(f"Average LPIPS: {avg_lpips:.4f}")

Average LPIPS: 0.6937


### FID and KID

In [15]:
metrics = calculate_metrics(
    input1=real_dir,
    input2=generated_dir,
    cuda=True,
    fid=True,
    kid=True,
    input1_max_samples=image_count,
    input2_max_samples=image_count,
    kid_subset_size=image_count
)

print(f"FID: {metrics['frechet_inception_distance']:.4f}")
print(f"KID Mean: {metrics['kernel_inception_distance_mean']:.4f}")
print(f"KID Std: {metrics['kernel_inception_distance_std']:.4f}")

Creating feature extractor "inception-v3-compat" with features ['2048']
Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/weights-inception-2015-12-05-6726825d.pth
100%|██████████| 91.2M/91.2M [00:01<00:00, 64.6MB/s]
Extracting features from input1
Looking for samples non-recursivelty in "/kaggle/input/face-sketches-collection/small_hed-augmented_ffhq_dataset/test/photos" with extensions png,jpg,jpeg
Found 158 samples
Processing samples                                                         
Extracting features from input2
Looking for samples non-recursivelty in "/kaggle/working/generated_for_metrics" with extensions png,jpg,jpeg
Found 158 samples
Processing samples                                                         
Frechet Inception Distance: 182.23013944904272
                                                                                 

FID: 182.2301
KID Mean: 0.0856
KID Std: 0.0000


Kernel Inception Distance: 0.08563056029583903 ± 2.5697880129510814e-07
